In [6]:
using Pkg
Pkg.activate(".")

  Activating project at `~/Projects/2025-Tablacares_popgen/FST`


In [7]:
using PopGen
using CSV

Read in the all the loci

In [8]:
yft = PopGen.read("../data/YFT.snp.kinrm.pcrelate.gen")

┌ Info: 
│  /home/pdimens/Projects/2025-Tablacares_popgen/data/YFT.snp.kinrm.pcrelate.gen
│  formatting: delimiter = tab, loci = vertical
└  data: loci = 7910, samples = 417, populations = 6


PopData{Diploid, 7910 SNP loci}
  Samples: 417
  Populations: 6

In [9]:
newpops = ["ATL", "GOA","IVC","SEN","TX", "VZ"]
populations!(yft, newpops)
populations(yft, counts = true)

 Renaming unique populations



Row,population,count
,String,Int64
1,ATL,77
2,GOA,87
3,IVC,72
4,SEN,68
5,TX,31
6,VZ,82


Global summary stats

In [10]:
lxl_global = summary(yft, by = "locus")
CSV.write("global.nei.fst", lxl_global)

"global.nei.fst"

Perform a locus-by-locus FST (Hudson) against populations

In [11]:
fst_lxl = pairwisefst(yft, by="locus", method = Hudson) ;

CSV.write("locbyloc.hudson.fst", fst_lxl.results)

"locbyloc.hudson.fst"

Split the outlier and neutral data

In [12]:
f = open("../outliers/bscan_outflank.outliers")    
outlier_loc = [line for line in readlines(f)]
close(f)
outlier_loc

11-element Vector{String}:
 "Talbacares_contig_3974_68999"
 "Talbacares_contig_4875_8668"
 "Talbacares_contig_5081_16094"
 "Talbacares_contig_5081_299323"
 "Talbacares_contig_5081_299511"
 "Talbacares_contig_5081_299524"
 "Talbacares_lg_4_3319997"
 "Talbacares_lg_4_3951863"
 "Talbacares_lg_11_7575958"
 "Talbacares_lg_17_3436257"
 "Talbacares_lg_22_772470"

Split outliers into their own Popdata object

In [13]:
yft_outliers = keep(yft, locus = outlier_loc)
yft_outliers

PopData{Diploid, 11 SNP loci}
  Samples: 417
  Populations: 6

Remove outliers (in-place) from original PopData

In [14]:
omit!(yft, locus = outlier_loc)
yft

PopData{Diploid, 7899 SNP loci}
  Samples: 417
  Populations: 6

FST on outlier data. Doing this first because it's faster and will precompile for the neutral data after.

In [ ]:
fst_outlier = pairwisefst(yft_outliers, method=Hudson, iterations=75000) ;
CSV.write("hudsonfst.outlier.csv", fst_outlier.results)

In [16]:
function Base.show(io::IO, data::PopGen.PairwiseFST)
    issymmetrical = size(data.results, 1) == size(data.results, 2)
    rwnames = issymmetrical ? names(data.results) : nothing
    show(
        io,
        round.(data.results, digits=4),
        show_row_number=false,
        row_number_column_label=" ",
        eltypes=false,
        row_labels=rwnames,
        title="Pairwise FST: " * data.method
    )
end

In [17]:
fst_outlier

Pairwise FST: Hudson (with p-values)
 Row │ ATL     GOA     IVC     SEN     TX      VZ  
─────┼─────────────────────────────────────────────
 ATL │ 0.0     0.0     0.0     0.0     0.0     0.0
 GOA │ 0.0302  0.0     0.0     0.0     0.0     0.0
 IVC │ 0.0373  0.1021  0.0     0.0     0.0     0.0
 SEN │ 0.0592  0.1493  0.016   0.0     0.0     0.0
  TX │ 0.0569  0.0048  0.1439  0.1946  0.0     0.0
  VZ │ 0.0235  0.0251  0.0995  0.129   0.0287  0.0

FST on the neutral data

In [ ]:
fst_neutral = pairwisefst(yft, method = Hudson, iterations = 75000) ;
CSV.write("hudsonfst.neutral.csv", fst_neutral.results)